In [1]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
def tokenize_word(word):
    return tokenizer.tokenize(word, add_special_tokens=False)

In [2]:
def get_boundaries(tokens, word):
    """Return a set of character positions (1‑based) where a boundary occurs."""
    pos = 0
    boundaries = set()
    for tok in tokens:
        pos += len(tok)
        if pos < len(word):
            boundaries.add(pos)
    return boundaries
gold = []
with open("gold_segmentation.txt", "r", encoding="utf-8") as f:
    for line in f:
        word, seg = line.strip().split("\t")
        parts = seg.split('+')
        pos = 0
        gold_boundaries = set()
        for part in parts:
            pos += len(part)
            if pos < len(word):
                gold_boundaries.add(pos)
        gold.append((word, gold_boundaries))

def evaluate_baseline(tokenizer_func, name):
    total_precision = 0.0
    total_recall = 0.0
    total_f1 = 0.0
    for word, gold_bounds in gold:
        tokens = tokenizer_func(word)
        pred_bounds = get_boundaries(tokens, word)

        intersect = gold_bounds & pred_bounds
        prec = len(intersect) / len(pred_bounds) if pred_bounds else 0.0
        rec = len(intersect) / len(gold_bounds) if gold_bounds else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        total_precision += prec
        total_recall += rec
        total_f1 += f1
    avg_prec = total_precision / len(gold)
    avg_rec = total_recall / len(gold)
    avg_f1 = total_f1 / len(gold)
    print(f"{name}: P={avg_prec:.4f}, R={avg_rec:.4f}, F1={avg_f1:.4f}")
evaluate_baseline(lambda w: tokenizer.tokenize(w, add_special_tokens=False), "IndicBERT")

IndicBERT: P=0.1355, R=0.2633, F1=0.1725


In [5]:
from morfessor import MorfessorIO
io = MorfessorIO()
model = io.read_binary_model_file("model.baseline.bin")
tokens = model.segment(word)[0]
tokens

'सक'

In [6]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor()
sp.load("sp_bpe.model")
tokens = sp.encode_as_pieces(word) 

In [7]:
tokens

['▁सकिन्छ']

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

trainer = trainers.BpeTrainer(
    vocab_size=16000,
    min_frequency=2,
    special_tokens=["<unk>", "<s>", "</s>", "<pad>"],
)

with open("dataset_ne/ne.txt", "r", encoding="utf-8") as f:
    tokenizer.train_from_iterator(f, trainer=trainer)

tokenizer.save("baseline_byte_bpe.json")
print("Byte-level BPE trained and saved.")

In [7]:
import sentencepiece as spm
from tokenizers import Tokenizer
from morfessor import MorfessorIO

# Load models
tokenizer_byte = Tokenizer.from_file("baseline_byte_bpe.json")

sp_bpe = spm.SentencePieceProcessor()
sp_bpe.Load("sp_bpe.model")

sp_unigram = spm.SentencePieceProcessor()
sp_unigram.Load("sp_unigram.model")

io = MorfessorIO()
model_morf =  io.read_binary_model_file("model.baseline.bin")

# Tokenizer functions
def tok_byte(word):
    return tokenizer_byte.encode(word).tokens

def tok_sp_bpe(word):
    return sp_bpe.EncodeAsPieces(word)

def tok_sp_unigram(word):
    return sp_unigram.EncodeAsPieces(word)

def tok_morfessor(word):
    # Morfessor returns (segmentation, logprob); we take the segmentation list
    return model_morf.segment(word)[0]

# IndicBERT already defined
from transformers import AutoTokenizer
tokenizer_ind = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
def tok_indic(word):
    return tokenizer_ind.tokenize(word, add_special_tokens=False)

In [8]:
evaluate_baseline(tok_byte, "Byte-level BPE")
evaluate_baseline(tok_sp_bpe, "SentencePiece BPE")
evaluate_baseline(tok_sp_unigram, "SentencePiece Unigram")
evaluate_baseline(tok_morfessor, "Morfessor")
evaluate_baseline(tok_indic, "IndicBERT (pretrained)")

Byte-level BPE: P=0.0604, R=0.0678, F1=0.0614
SentencePiece BPE: P=0.0000, R=0.0000, F1=0.0000
SentencePiece Unigram: P=0.0430, R=0.0427, F1=0.0415
Morfessor: P=0.0823, R=0.3195, F1=0.1259
IndicBERT (pretrained): P=0.1355, R=0.2633, F1=0.1725


In [1]:
import tiny_LLM_scratch_with_tokenizer
tokenizer = tiny_LLM_scratch_with_tokenizer.PyNepBPETokenizer()

In [2]:
from collections import Counter
word_freqs = Counter()
chunk_size = 100_000  # characters per chunk

with open("dataset_ne/ne.txt", "r", encoding="utf-8") as f:
    while True:
        chunk = f.read(chunk_size)
        if not chunk:
            break
        
        # Normalize chunk
        try:
            normalized = tokenizer.normalize(chunk)
            words = normalized.split()
            word_freqs.update(words)
            print(f"Processed chunk, total words so far: {sum(word_freqs.values())}")
        except Exception as e:
            print(f"Error on chunk: {e}")
            break

print(f"\nTotal unique words: {len(word_freqs)}")
print(f"Total words: {sum(word_freqs.values())}")

lengths = sorted((len(w) for w in word_freqs), reverse=True)
print("Longest words (chars):", lengths[:10])
print("Words >200 chars:", sum(l > 200 for l in lengths))

Processed chunk, total words so far: 15343
Processed chunk, total words so far: 30859
Processed chunk, total words so far: 46923
Processed chunk, total words so far: 62356
Processed chunk, total words so far: 77906
Processed chunk, total words so far: 93802
Processed chunk, total words so far: 109645
Processed chunk, total words so far: 124997
Processed chunk, total words so far: 140217
Processed chunk, total words so far: 155551
Processed chunk, total words so far: 170852
Processed chunk, total words so far: 186271
Processed chunk, total words so far: 201702
Processed chunk, total words so far: 217060
Processed chunk, total words so far: 232601
Processed chunk, total words so far: 247968
Processed chunk, total words so far: 263317
Processed chunk, total words so far: 278834
Processed chunk, total words so far: 294188
Processed chunk, total words so far: 309331
Processed chunk, total words so far: 324691
Processed chunk, total words so far: 340543
Processed chunk, total words so far: 3

In [1]:
import unicodedata
from collections import Counter
import heapq

def normalize_python(s, folding_table=None):
    s = unicodedata.normalize('NFC', s)
    if folding_table:
        s = ''.join(folding_table.get(c, c) for c in s)
    s = s.replace('\u200D', '')  # remove ZWJ
    return s

# Keep a heap of the 5 longest words (negative length for min‑heap)
longest = []  # will store (-len, word)

with open("dataset_ne/ne.txt", "r", encoding="utf-8") as f:
    for line in f:
        normalized_line = normalize_python(line)
        # Split into words – you may want a smarter tokenizer
        for word in normalized_line.split():
            # Push to heap if it's among the longest
            if len(longest) < 5:
                heapq.heappush(longest, (len(word), word))
            elif len(word) > longest[0][0]:
                heapq.heapreplace(longest, (len(word), word))

# Get the top 5 in descending order
longest_words = sorted(longest, key=lambda x: -x[0])

for i, (length, w) in enumerate(longest_words):
    print(f"\n{'='*60}")
    print(f"Long word #{i+1} (len={length})")
    print(f"{'='*60}")
    print(f"First 150 chars:  {w[:150]!r}")
    print(f"Last 150 chars:   {w[-150:]!r}")

    devanagari = sum(1 for c in w if '\u0900' <= c <= '\u097F')
    latin = sum(1 for c in w if c.isascii() and c.isalpha())
    digits = sum(1 for c in w if c.isdigit())
    punctuation = sum(1 for c in w if not c.isalnum())
    other = len(w) - devanagari - latin - digits - punctuation

    print(f"\nBreakdown:")
    print(f"  Devanagari: {devanagari:6} ({100*devanagari/len(w):.1f}%)")
    print(f"  Latin:      {latin:6} ({100*latin/len(w):.1f}%)")
    print(f"  Digits:     {digits:6} ({100*digits/len(w):.1f}%)")
    print(f"  Punctuation:{punctuation:6} ({100*punctuation/len(w):.1f}%)")
    print(f"  Other:      {other:6} ({100*other/len(w):.1f}%)")


Long word #1 (len=4577)
First 150 chars:  '.ExtraTorrent,,,,>,,,,Categories,,,,>,,,,Movies,,,,torrents,,,,>,,,,Dubbed,,,,Movies,,,,torrentsDownload,,,Fast,,,Safe,,,Anonymousmovies,,,,software,,'
Last 150 chars:   ',,Adventure,,add,,G.IJoe,,Retaliation,,(2013),,Extended,,Action,,Cut,,1080p,,BluRay,,x264,,(Dual,,Audio),,{English+,,Hindi,,},,-=YAKMJY=-,,Aug,,2013,,'

Breakdown:
  Devanagari:      0 (0.0%)
  Latin:        2376 (51.9%)
  Digits:        445 (9.7%)
  Punctuation:  1756 (38.4%)
  Other:           0 (0.0%)

Long word #2 (len=1807)
First 150 chars:  'Bitch)&amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp'
Last 150 chars:   'mp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;amp;gt;'

Breakdown:
  Devanagari:      0 (0.0%)
  Latin:        1350 (74.7%)
  Digits:          0 (0.0%)
  Punctuation: 

In [2]:
import unicodedata
import heapq

def normalize_python(s, folding_table=None):
    s = unicodedata.normalize('NFC', s)
    if folding_table:
        s = ''.join(folding_table.get(c, c) for c in s)
    s = s.replace('\u200D', '')   # remove ZWJ, keep ZWNJ
    return s

def is_devanagari_word(w):
    return any('\u0900' <= c <= '\u097F' for c in w)

longest = []   # (length, word) heap, keeps the shortest among the top 5

with open("dataset_ne/ne.txt", "r", encoding="utf-8") as f:
    for line in f:
        normalized_line = normalize_python(line)
        for word in normalized_line.split():
            if not is_devanagari_word(word):
                continue
            if len(longest) < 5:
                heapq.heappush(longest, (len(word), word))
            elif len(word) > longest[0][0]:
                heapq.heapreplace(longest, (len(word), word))

# Sort descending by length
longest_words = sorted(longest, key=lambda x: -x[0])

for i, (length, w) in enumerate(longest_words):
    print(f"\n{'='*60}")
    print(f"Longest Nepali word #{i+1} (len={length})")
    print(f"{'='*60}")
    print(f"First 150 chars:  {w[:150]!r}")
    print(f"Last 150 chars:   {w[-150:]!r}")

    dev = sum(1 for c in w if '\u0900' <= c <= '\u097F')
    lat = sum(1 for c in w if c.isascii() and c.isalpha())
    dig = sum(1 for c in w if c.isdigit())
    punc = sum(1 for c in w if not c.isalnum())
    other = len(w) - dev - lat - dig - punc

    print(f"\nBreakdown:")
    print(f"  Devanagari: {dev:6} ({100*dev/len(w):.1f}%)")
    print(f"  Latin:      {lat:6} ({100*lat/len(w):.1f}%)")
    print(f"  Digits:     {dig:6} ({100*dig/len(w):.1f}%)")
    print(f"  Punctuation:{punc:6} ({100*punc/len(w):.1f}%)")
    print(f"  Other:      {other:6} ({100*other/len(w):.1f}%)")


Longest Nepali word #1 (len=386)
First 150 chars:  'कोरियाकोसोभोकुवेतकिर्गिस्तानलाओसलाटवियालेबनानलेसोथोलाइबेरियालिबियालिएखटेन्स्टाइनलिथुआनियालक्जेम्बर्गम्यासिडोनियामडागास्करमालवीमलेसियामाल्दिभ्समालीमाल्'
Last 150 chars:   '्रोमोरक्कोमोजाम्बिकम्यानमारकोनामिबियानाउरूनेपालनेदरल्यान्ड्सन्यूजील्याण्डनिकारागुवानाइजरनाइजेरियाउत्तरीमारिआनाद्वीपनर्वेओमानपाकिस्तानपलाउप्यालेस्टाइन,'

Breakdown:
  Devanagari:    376 (97.4%)
  Latin:           9 (2.3%)
  Digits:          0 (0.0%)
  Punctuation:   162 (42.0%)
  Other:        -161 (-41.7%)

Longest Nepali word #2 (len=386)
First 150 chars:  'कोरियाकोसोभोकुवेतकिर्गिस्तानलाओसलाटवियालेबनानलेसोथोलाइबेरियालिबियालिएखटेन्स्टाइनलिथुआनियालक्जेम्बर्गम्यासिडोनियामडागास्करमालवीमलेसियामाल्दिभ्समालीमाल्'
Last 150 chars:   '्रोमोरक्कोमोजाम्बिकम्यानमारकोनामिबियानाउरूनेपालनेदरल्यान्ड्सन्यूजील्याण्डनिकारागुवानाइजरनाइजेरियाउत्तरीमारिआनाद्वीपनर्वेओमानपाकिस्तानपलाउप्यालेस्टाइन,'

Breakdown:
  Devanagari:    376 (97.4%)
  Latin:           9 (2.3%)
  Digits:          0 (0

In [3]:
import unicodedata
import re

def normalize_python(s, folding_table=None):
    s = unicodedata.normalize('NFC', s)
    if folding_table:
        s = ''.join(folding_table.get(c, c) for c in s)
    s = s.replace('\u200D', '')   # remove ZWJ
    return s

def is_noise_line(line):
    """Return True if the line should be removed."""
    line = line.strip()
    if len(line) < 3:
        return True

    # 1. Remove torrent / download spam
    spam_keywords = ['torrent', 'download', 'dubbed', 'bluray', 'x264', '1080p', '720p', 'extratorrent']
    if any(kw in line.lower() for kw in spam_keywords):
        return True

    # 2. Remove lines that are mostly HTML entities (e.g., &amp;)
    if line.count('&amp;') > 3 or line.count('&lt;') > 3:
        return True

    # 3. Count Devanagari characters
    dev_count = sum(1 for c in line if '\u0900' <= c <= '\u097F')
    # If no Devanagari, keep only if the line looks like a meaningful sentence (e.g., contains spaces and alphabetic chars)
    if dev_count == 0:
        # Keep only if it has at least 3 alphabetic chars and some spaces (likely proper English text)
        alpha = sum(1 for c in line if c.isalpha())
        if alpha < 5 or ' ' not in line:
            return True
        # You could also keep Latin lines, but here we'll drop them because we want mainly Nepali.
        # If you want to keep English sentences, set a threshold.
        return True   # drop all zero-Devanagari lines

    # 4. If line has Devanagari, but very low ratio, maybe it's mixed junk
    text_len = len(line) - line.count(' ')  # ignore spaces
    if text_len > 0 and dev_count / text_len < 0.2:
        return True

    # 5. Lines that are all punctuation / digits
    punct_digit = sum(1 for c in line if not c.isalnum() and not c.isspace())
    if punct_digit > 0.8 * len(line):
        return True

    # 6. Extremely long words without spaces (concatenated lists)
    # Split by spaces; if any word > 100 chars and has Devanagari, we may want to split it
    # Here we simply drop lines with such words (or you can implement splitting)
    words = line.split()
    for w in words:
        if len(w) > 100 and dev_count > 0:
            return True

    return False

def fix_concatenated_words(line):
    """Try to insert spaces between known country names (if you have a list)."""
    return line

def clean_corpus(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:
        for line in fin:
            line = normalize_python(line)
            if is_noise_line(line):
                continue
            line = fix_concatenated_words(line)
            fout.write(line)

clean_corpus("dataset_ne/ne.txt", "dataset_ne/ne_cleaned.txt")

In [4]:
import unicodedata
import heapq

def normalize_python(s, folding_table=None):
    s = unicodedata.normalize('NFC', s)
    if folding_table:
        s = ''.join(folding_table.get(c, c) for c in s)
    s = s.replace('\u200D', '')   # remove ZWJ, keep ZWNJ
    return s

def is_devanagari_word(w):
    return any('\u0900' <= c <= '\u097F' for c in w)

longest = []   # (length, word) heap, keeps the shortest among the top 5

with open("dataset_ne/ne_cleaned.txt", "r", encoding="utf-8") as f:
    for line in f:
        normalized_line = normalize_python(line)
        for word in normalized_line.split():
            if not is_devanagari_word(word):
                continue
            if len(longest) < 5:
                heapq.heappush(longest, (len(word), word))
            elif len(word) > longest[0][0]:
                heapq.heapreplace(longest, (len(word), word))

# Sort descending by length
longest_words = sorted(longest, key=lambda x: -x[0])

for i, (length, w) in enumerate(longest_words):
    print(f"\n{'='*60}")
    print(f"Longest Nepali word #{i+1} (len={length})")
    print(f"{'='*60}")
    print(f"First 150 chars:  {w[:150]!r}")
    print(f"Last 150 chars:   {w[-150:]!r}")

    dev = sum(1 for c in w if '\u0900' <= c <= '\u097F')
    lat = sum(1 for c in w if c.isascii() and c.isalpha())
    dig = sum(1 for c in w if c.isdigit())
    punc = sum(1 for c in w if not c.isalnum())
    other = len(w) - dev - lat - dig - punc

    print(f"\nBreakdown:")
    print(f"  Devanagari: {dev:6} ({100*dev/len(w):.1f}%)")
    print(f"  Latin:      {lat:6} ({100*lat/len(w):.1f}%)")
    print(f"  Digits:     {dig:6} ({100*dig/len(w):.1f}%)")
    print(f"  Punctuation:{punc:6} ({100*punc/len(w):.1f}%)")
    print(f"  Other:      {other:6} ({100*other/len(w):.1f}%)")


Longest Nepali word #1 (len=99)
First 150 chars:  'गौशाला-बत्तिसपुतली-मैतिदेवी-पुतलीसडक-घण्टाघर-शहिदगेट-त्रिपुरेश्वर-थापाथली-पुल्चोक-मंगलबजार-ग्वार्को'
Last 150 chars:   'गौशाला-बत्तिसपुतली-मैतिदेवी-पुतलीसडक-घण्टाघर-शहिदगेट-त्रिपुरेश्वर-थापाथली-पुल्चोक-मंगलबजार-ग्वार्को'

Breakdown:
  Devanagari:     89 (89.9%)
  Latin:           0 (0.0%)
  Digits:          0 (0.0%)
  Punctuation:    44 (44.4%)
  Other:         -34 (-34.3%)

Longest Nepali word #2 (len=98)
First 150 chars:  '(रातडाँडा–हर्पूकोट–नयाँपोखरा–दोहली–इस्मामैदान–दोवाटा–सिरार–चोरकाटे–बुल्म–हुलाके–राहले–पालुखा–छल्दी'
Last 150 chars:   '(रातडाँडा–हर्पूकोट–नयाँपोखरा–दोहली–इस्मामैदान–दोवाटा–सिरार–चोरकाटे–बुल्म–हुलाके–राहले–पालुखा–छल्दी'

Breakdown:
  Devanagari:     85 (86.7%)
  Latin:           0 (0.0%)
  Digits:          0 (0.0%)
  Punctuation:    50 (51.0%)
  Other:         -37 (-37.8%)

Longest Nepali word #3 (len=97)
First 150 chars:  '"https://ne.wikipedia.org/w/index.php?title=कृष्ण_चेतनाका_लागि_अन्तर्राष्ट्रिय_समाज&oldid=67

In [8]:
import tiny_LLM_scratch_with_tokenizer
tokenizer = tiny_LLM_scratch_with_tokenizer.PyNepBPETokenizer()
from collections import Counter
word_freqs = Counter()
chunk_size = 100_000  
with open("dataset_ne/ne_cleaned.txt", "r", encoding="utf-8") as f:
    while True:
        chunk = f.read(chunk_size)
        if not chunk:
            break
        
        # Normalize chunk
        try:
            normalized = tokenizer.normalize(chunk)
            words = normalized.split()
            word_freqs.update(words)
            print(f"Processed chunk, total words so far: {sum(word_freqs.values())}")
        except Exception as e:
            print(f"Error on chunk: {e}")
            break

print(f"\nTotal unique words: {len(word_freqs)}")
print(f"Total words: {sum(word_freqs.values())}")

lengths = sorted((len(w) for w in word_freqs), reverse=True)
print("Longest words (chars):", lengths[:10])
print("Words >200 chars:", sum(l > 200 for l in lengths))

Processed chunk, total words so far: 15349
Processed chunk, total words so far: 30865
Processed chunk, total words so far: 46952
Processed chunk, total words so far: 62381
Processed chunk, total words so far: 77933
Processed chunk, total words so far: 93830
Processed chunk, total words so far: 109687
Processed chunk, total words so far: 125034
Processed chunk, total words so far: 140293
Processed chunk, total words so far: 155613
Processed chunk, total words so far: 170929
Processed chunk, total words so far: 186343
Processed chunk, total words so far: 201792
Processed chunk, total words so far: 217124
Processed chunk, total words so far: 232672
Processed chunk, total words so far: 248005
Processed chunk, total words so far: 263397
Processed chunk, total words so far: 278903
Processed chunk, total words so far: 294238
Processed chunk, total words so far: 309400
Processed chunk, total words so far: 324850
Processed chunk, total words so far: 340596
Processed chunk, total words so far: 3

In [ ]:
import tiny_llm_scratch_with_tokenizer as tok
print(tok.__file__)
/home/lang-chain/Documents/Astra_agentic_RAG/.venv/lib/python3.11/site-packages/tiny_llm_scratch_with_tokenizer/__init__.py

/home/lang-chain/Documents/Astra_agentic_RAG/.venv/lib/python3.11/site-packages/tiny_llm_scratch_with_tokenizer/__init__.py


In [ ]:
import tiny_llm_scratch_with_tokenizer as tok
print(dir(tok.PyNepBPETokenizer))

['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'add_dfa_transition', 'add_paradigm', 'decode', 'encode', 'get_token_surface', 'initialize_vocab', 'normalize', 'train_bpe', 'train_from_text', 'vocab_size']

['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'add_dfa_transition', 'add_paradigm', 'decode', 'encode', 'get_token_surface', 'initialize_vocab', 'normalize', 'train_bpe', 'train_from_text', 'vocab_size']


In [ ]:
import tiny_llm_scratch_with_tokenizer as tok

tokenizer = tok.PyNepBPETokenizer()

# Test 1: Check that the new methods exist
print("=== Method Existence Test ===")
print(f"Has dfa_has_transition: {hasattr(tokenizer, 'dfa_has_transition')}")
print(f"Has dfa_tokenize_debug: {hasattr(tokenizer, 'dfa_tokenize_debug')}")
print(f"Has vocab_contains: {hasattr(tokenizer, 'vocab_contains')}")

# Test 2: Basic DFA test
print("\n=== Basic DFA Test ===")
tokenizer.add_dfa_transition(0, 'स', 1, True)
result = tokenizer.dfa_has_transition(0, 'स')
print(f"After adding (0, 'स')->1: {result}")

# Test 3: Check what methods ARE available
print("\n=== All Available Methods ===")
methods = [m for m in dir(tokenizer) if not m.startswith('_')]
for m in methods:
    print(f"  {m}")


    

=== Method Existence Test ===
Has dfa_has_transition: False
Has dfa_tokenize_debug: False
Has vocab_contains: False

=== Basic DFA Test ===


AttributeError: 'builtins.PyNepBPETokenizer' object has no attribute 'dfa_has_transition'

import tiny_llm_scratch_with_tokenizer as tok
print(tok.__file__)

In [ ]:
import tiny_llm_scratch_with_tokenizer as tok

print("1. Initializing Tokenizer...")
tokenizer = tok.PyNepBPETokenizer()

# =====================================================================
# STEP 1: Build the Akshara DFA (Devanagari Syllable Parser)
# =====================================================================
print("2. Building Devanagari DFA...")
vowels = "अआइईउऊएऐओऔ"
consonants = "कखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसह"
matras = "ािीुूेैोौ्"  # Includes Halant (्) at the end
dependent_vowels = matras[:-1]  # ा ि ी ू े ै ो ौ
halant = matras[-1]  # ्

# State 0: Initial
# State 1: Got a consonant or standalone vowel (Accepting)
# State 2: Got a matra/vowel sign after consonant (Accepting)  
# State 3: Got halant after consonant (NOT accepting - half-form)
# State 4: Got consonant after halant = conjunct (Accepting)
# State 5: Got matra after conjunct (Accepting)

# Standalone Vowels
for v in vowels:
    tokenizer.add_dfa_transition(0, v, 1, True)

# Consonants
for c in consonants:
    # Standalone consonant
    tokenizer.add_dfa_transition(0, c, 1, True)
    
    # Consonant + Dependent Vowel (e.g., का, कि, के)
    for m in dependent_vowels:
        tokenizer.add_dfa_transition(1, m, 2, True)
    
    # Consonant + Halant (half-form, not accepting)
    tokenizer.add_dfa_transition(1, halant, 3, False)
    
    # Halant + Consonant = Conjunct base (e.g., क्क, स्त)
    for c2 in consonants:
        tokenizer.add_dfa_transition(3, c2, 4, True)
        
        # Conjunct + Dependent Vowel (e.g., स्ते, क्का)
        for m in dependent_vowels:
            tokenizer.add_dfa_transition(4, m, 5, True)
        
        # Conjunct + Halant (for triple conjuncts like क्क्त)
        tokenizer.add_dfa_transition(4, halant, 3, False)

# =====================================================================
# STEP 2: Initialize Base Vocabulary (COMPREHENSIVE)
# =====================================================================
print("3. Initializing base vocabulary...")

base_aksharas = []

# Individual vowels
base_aksharas.extend(list(vowels))

# Individual consonants  
base_aksharas.extend(list(consonants))

# Consonant + dependent vowel combinations (e.g., का, कि, ..., हौ)
for c in consonants:
    for m in dependent_vowels:
        base_aksharas.append(c + m)

# Consonant + halant (half-forms)
for c in consonants:
    base_aksharas.append(c + halant)

# Common two-consonant conjuncts
common_pairs = [
    "क्क", "क्ख", "क्ग", "क्त", "क्त्र", "क्थ", "क्द", "क्ध", "क्न", "क्प", 
    "क्फ", "क्ब", "क्म", "क्य", "क्र", "क्ल", "क्व",
    "ग्ग", "ग्घ", "ग्ध", "ग्न", "ग्म", "ग्य", "ग्र", "ग्ल",
    "च्च", "च्छ", "च्ज", "च्ञ", "च्य",
    "ज्ज", "ज्झ", "ज्ञ", "ज्य",
    "ट्ट", "ट्ठ", "ट्ण", "ट्म", "ट्य",
    "ड्ड", "ड्ढ", "ड्ण", "ड्म",
    "त्त", "त्त्व", "त्थ", "त्न", "त्प", "त्म", "त्य", "त्र", "त्ल",
    "द्ग", "द्घ", "द्द", "द्ध", "द्ध्र", "द्न", "द्ब", "द्भ", "द्म", "द्य", "द्र", "द्व",
    "ध्न", "ध्म", "ध्य", "ध्र", "ध्व",
    "न्न", "न्त", "न्त्र", "न्त्य", "न्ध", "न्ध्र", "न्न", "न्म", "न्य",
    "प्त", "प्न", "प्प", "प्म", "प्य", "प्र", "प्ल",
    "ब्ज", "ब्ब", "ब्भ", "ब्य",
    "म्न", "म्प", "म्ब", "म्भ", "म्म", "म्य", "म्र",
    "य्य",
    "र्क", "र्ख", "र्ग", "र्घ", "र्च", "र्छ", "र्ज", "र्झ", "र्ञ", "र्ट", "र्ठ",
    "र्ड", "र्ढ", "र्त", "र्त्र", "र्थ", "र्द", "र्ध", "र्ध्र", "र्न", "र्प",
    "र्फ", "र्ब", "र्भ", "र्म", "र्य", "र्र", "र्ल", "र्व",
    "ल्क", "ल्ख", "ल्ग", "ल्घ", "ल्च", "ल्ज", "ल्झ", "ल्ट", "ल्ठ", "ल्ड", "ल्ढ",
    "ल्त", "ल्थ", "ल्द", "ल्ध", "ल्न", "ल्प", "ल्फ", "ल्ब", "ल्भ", "ल्म", "ल्य", "ल्ल",
    "व्य", "व्व",
    "श्च", "श्छ", "श्झ", "श्ञ", "श्ठ", "श्म", "श्य", "श्र", "श्ल",
    "ष्क", "ष्ख", "ष्ग", "ष्ट", "ष्ठ", "ष्ण", "ष्प", "ष्फ", "ष्म", "ष्य",
    "स्क", "स्ख", "स्त", "स्त्र", "स्थ", "स्न", "स्प", "स्फ", "स्म", "स्य",
    "ह्ण", "ह्म", "ह्य", "ह्ल", "ह्व",
]

# Add conjuncts and conjunct+matra forms
for conj in common_pairs:
    base_aksharas.append(conj)
    for m in dependent_vowels:
        base_aksharas.append(conj + m)
    base_aksharas.append(conj + halant)  # For triple conjuncts

# Seed morphemes
seed_morphemes = ["कर", "मा", "को", "ले", "हो", "छ", "गर", "छन्", "छु"]

punctuation = ["।", "॥", ",", ".", " ", "\n", "!", "?", "।"]  # Danda included

# Morphological constraints
v_strict = ["को", "ले", "मा"]     # Postpositions - NEVER extend these
v_ambiguous = ["कर", "गर"]        # Verbs - Only merge if frequency > theta

tokenizer.initialize_vocab(
    aksharas=base_aksharas,
    seed_morphemes=seed_morphemes,
    punctuation=punctuation,
    v_strict=v_strict,
    v_ambiguous=v_ambiguous
)
print(f"   Base Vocab Size: {tokenizer.vocab_size()}")

# =====================================================================
# STEP 3: Train BPE on Corpus
# =====================================================================
print("4. Training Constrained BPE...")

corpus = [
    "नमस्ते नेपालमा स्वागत छ",
    "काठमाडौं नेपालको राजधानी हो",
    "नेपाल एक सुन्दर देश हो",
    "हिमालय नेपालको उत्तरमा छ",
    "गंगा नदी नेपालबाट बग्छ",
    "नेपालमा धेरै भाषाहरू बोलिन्छ",
    "काठमाडौंमा धेरै मन्दिरहरू छन्",
    "बुद्ध नेपालमा जन्मिएका थिए",
    "नेपाली भाषा सुन्दर छ",
    "म नेपालमा बस्छु",
    "काठमाडौं सहरमा धेरै मानिस बस्छन्",
    "नेपालको झण्डा लाल र सेतो छ",
]

final_size = tokenizer.train_from_text(
    texts=corpus, 
    vocab_budget=500, 
    theta=5
)
print(f"   Trained Vocab Size: {final_size}")

# =====================================================================
# STEP 4: Test Encode / Decode
# =====================================================================
print("\n5. Testing Encode/Decode Roundtrip...")

test_sentences = [
    "नमस्ते नेपाल",
    "काठमाडौं नेपालको राजधानी हो।",
    "म नेपालमा बस्छु",
    "स्ते",  # Test conjunct+matra
    "काली",  # Test consonant+matra
]

all_pass = True
for text in test_sentences:
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)
    normalized = tokenizer.normalize(text)
    
    passed = decoded == normalized
    if not passed:
        all_pass = False
    
    status = "✅" if passed else "❌"
    print(f"{status} Original : {text}")
    print(f"   Normalized: {normalized}")
    print(f"   Token IDs: {ids}")
    print(f"   Decoded  : {decoded}")
    if not passed:
        for i, (a, b) in enumerate(zip(decoded, normalized)):
            if a != b:
                print(f"   Diff at pos {i}: got '{a}' (U+{ord(a):04X}), expected '{b}' (U+{ord(b):04X})")
        if len(decoded) != len(normalized):
            print(f"   Length mismatch: decoded {len(decoded)}, normalized {len(normalized)}")
    print()

print(f"\n{'='*50}")
print(f"ALL TESTS PASSED: {all_pass}")



1. Initializing Tokenizer...
2. Building Devanagari DFA...
3. Initializing base vocabulary...
   Base Vocab Size: 2683
4. Training Constrained BPE...
   Trained Vocab Size: 2683

5. Testing Encode/Decode Roundtrip...
❌ Original : नमस्ते नेपाल
   Normalized: नमस्ते नेपाल
   Token IDs: [29, 34, 41, 25, 2458, 29, 30, 37]
   Decoded  : नमसत नपल
   Diff at pos 3: got 'त' (U+0924), expected '्' (U+094D)
   Diff at pos 4: got ' ' (U+0020), expected 'त' (U+0924)
   Diff at pos 5: got 'न' (U+0928), expected 'े' (U+0947)
   Diff at pos 6: got 'प' (U+092A), expected ' ' (U+0020)
   Diff at pos 7: got 'ल' (U+0932), expected 'न' (U+0928)
   Length mismatch: decoded 8, normalized 12

❌ Original : काठमाडौं नेपालको राजधानी हो।
   Normalized: काठमाडौं नेपालको राजधानी हो।
   Token IDs: [10, 21, 34, 22, 2458, 29, 30, 37, 10, 2458, 36, 17, 28, 29, 2458, 42, 2425]
   Decoded  : कठमड नपलक रजधन ह।
   Diff at pos 1: got 'ठ' (U+0920), expected 'ा' (U+093E)
   Diff at pos 2: got 'म' (U+092E), expected 'ठ' (U+09